In [ ]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

from bs4 import BeautifulSoup
import io

import csv

import time

In [ ]:
#### setting dictionaries - mappings 

In [24]:
import os
proj_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebooks/ -> project root
data_dir = os.path.join(proj_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [26]:
import os, sys
print(data_dir)
print("cwd:", os.getcwd())
print("src exists:", os.path.exists(os.path.join(os.getcwd(), "src")))
print("sys.path[0]:", sys.path[0])

c:\Users\jachy\Desktop\Data-Processing-in-Python---Project\data\raw
cwd: c:\Users\jachy\Desktop\Data-Processing-in-Python---Project\notebooks
src exists: False
sys.path[0]: c:\Users\jachy\Desktop\Data-Processing-in-Python---Project


In [113]:
### chmi weather stations - data processing

df_chmi_stat = pd.read_csv(os.path.join(data_dir, "chmi_stations_metadata.csv"))

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

wsi_to_drop = [
    '0-203-0-11201020001', #Praha, Vinohrady - Flora	
    '0-203-0-11202007001', #Praha, Suchdol
    '0-203-0-11105048001', #Praha, Zadní Kopanina
    '0-203-0-11201020003' #Praha, Chodov
    ]

df_chmi_stat = df_chmi_stat[~df_chmi_stat['WSI'].isin(wsi_to_drop)]

df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [114]:
df_chmi_stat

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999-12-31 23:59:00+00:00
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999-12-31 23:59:00+00:00
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999-12-31 23:59:00+00:00
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999-12-31 23:59:00+00:00
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999-12-31 23:59:00+00:00
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999-12-31 23:59:00+00:00
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999-12-31 23:59:00+00:00
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999-12-31 23:59:00+00:00
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999-12-31 23:59:00+00:00


In [115]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

with open(os.path.join(data_dir, "wsi_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(wsi_dict.items())

In [43]:
### chmi weather variables

### filtering only needed ones

df_chmi_vars = pd.read_csv(os.path.join(data_dir, "chmi_variables_metadata.csv"))

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

with open(os.path.join(data_dir, "chmi_vars_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(chmi_vars_dict.items())

In [ ]:
### chmi weather 10min data

df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))

# it gives flag warning, but i assume flag is not of interest


C:\Users\jachy\AppData\Local\Temp\ipykernel_26524\3649673168.py:3: DtypeWarning: Columns (0: FLAG) have mixed types. Specify dtype option on import or set low_memory=False.
  df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))


In [47]:
df_weather.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-20000-0-11518,Casmax,2025-01-01T00:00:00Z,273.0,NaN,0.0,0-20000-0-11518,2025,1
1,0-20000-0-11518,Casmax,2025-01-01T00:10:00Z,32.0,NaN,0.0,0-20000-0-11518,2025,1
2,0-20000-0-11518,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1
3,0-20000-0-11518,Casmax,2025-01-01T00:30:00Z,283.0,NaN,0.0,0-20000-0-11518,2025,1
4,0-20000-0-11518,Casmax,2025-01-01T00:40:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1


In [52]:
df_weather['FLAG'].isna().sum()

np.int64(5023823)

In [51]:
df_weather['FLAG'].notna().sum()

np.int64(21577)

In [53]:
df_weather['FLAG'].unique()

<StringArray>
[nan, 'Z', 'S', 'N', 'P']
Length: 5, dtype: str

In [55]:
df_weather['QUALITY'].unique()

array([0., 4., 3.])

In [61]:
### mapping variables names in weather data 
df_weather['ELEMENT_NAME'] = df_weather['ELEMENT'].map(chmi_vars_dict)

df_weather['WSI_NAME'] = df_weather['WSI'].map(wsi_dict)


In [62]:
df_weather

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH,ELEMENT_NAME,WSI_NAME
0,0-20000-0-11518,Casmax,2025-01-01T00:00:00Z,273.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně"
1,0-20000-0-11518,Casmax,2025-01-01T00:10:00Z,32.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně"
2,0-20000-0-11518,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně"
3,0-20000-0-11518,Casmax,2025-01-01T00:30:00Z,283.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně"
4,0-20000-0-11518,Casmax,2025-01-01T00:40:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně"
...,...,...,...,...,...,...,...,...,...,...,...
5045395,0-203-0-11101007001,T,2025-12-31T23:10:00Z,-3.7,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy"
5045396,0-203-0-11101007001,T,2025-12-31T23:20:00Z,-3.6,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy"
5045397,0-203-0-11101007001,T,2025-12-31T23:30:00Z,-3.6,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy"
5045398,0-203-0-11101007001,T,2025-12-31T23:40:00Z,-3.6,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy"


In [63]:
### add metadata about stations

stations_meta = pd.read_csv(os.path.join(data_dir, "chmi_stations_metadata.csv"))

In [64]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION
0,0-20000-0-04030,ZIS04030,2015-01-01T00:00:00Z,3999-12-31T23:59:00Z,Reykjavik,-21.903905,64.127653,51.0
1,0-20000-0-11406,L3CHEB01,1863-10-01T00:00:00Z,1919-12-31T23:59:00Z,Cheb,12.362892,50.076212,458.0
2,0-20000-0-11406,L3CHEB01,1933-05-07T00:00:00Z,1938-04-30T23:59:00Z,Cheb,12.388900,50.073900,471.0
3,0-20000-0-11406,L3CHEB01,1943-06-01T00:00:00Z,1945-01-31T23:59:00Z,Cheb,12.388900,50.073900,471.0
4,0-20000-0-11406,L3CHEB01,1951-01-01T00:00:00Z,1960-12-31T23:59:00Z,Cheb,12.388900,50.073900,471.0
...,...,...,...,...,...,...,...,...
5534,0-203-0-42109005003,B4ZITK01,2015-09-16T00:00:00Z,3999-12-31T23:59:00Z,Žítková,17.867778,48.992500,705.0
5535,0-203-0-42109007001,B1PITI01,1944-10-01T00:00:00Z,1961-12-31T23:59:00Z,Pitín,17.890500,49.000900,342.0
5536,0-203-0-42109026001,B1LOPE01,1902-01-01T00:00:00Z,1902-12-31T23:59:00Z,Lopeník,17.792800,48.945800,672.0
5537,0-203-0-42109026001,B1LOPE01,1936-03-15T00:00:00Z,1946-06-30T23:59:00Z,Lopeník,17.792800,48.945800,672.0


In [65]:
stations_meta = stations_meta[stations_meta['WSI'].isin(wsi_dict)]

stations_meta["END_DATE_DT"] = pd.to_datetime(stations_meta["END_DATE"], utc=True, errors="coerce")

stations_meta = (
    stations_meta.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [66]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999-12-31 23:59:00+00:00
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999-12-31 23:59:00+00:00
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999-12-31 23:59:00+00:00
1425,0-203-0-11201020003,P1PCHO01,1997-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Chodov",14.500000,50.029400,297.00,3999-12-31 23:59:00+00:00
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999-12-31 23:59:00+00:00
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999-12-31 23:59:00+00:00
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999-12-31 23:59:00+00:00
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999-12-31 23:59:00+00:00
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999-12-31 23:59:00+00:00
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999-12-31 23:59:00+00:00


In [ ]:
stations_sel = stations_meta[["WSI", "FULL_NAME", "ELEVATION", "GEOGR1", "GEOGR2"]].copy()
stations_sel = stations_sel.rename(columns={
    "GEOGR1": "LON",
    "GEOGR2": "LAT"
})

df_weather["WSI"] = df_weather["WSI"].astype(str)
stations_sel["WSI"] = stations_sel["WSI"].astype(str)

In [73]:
df_weather_ext = df_weather.merge(stations_sel, on="WSI", how="left")


In [74]:
df_weather_ext

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH,ELEMENT_NAME,WSI_NAME,FULL_NAME,ELEVATION,LON,LAT
0,0-20000-0-11518,Casmax,2025-01-01T00:00:00Z,273.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
1,0-20000-0-11518,Casmax,2025-01-01T00:10:00Z,32.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
2,0-20000-0-11518,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
3,0-20000-0-11518,Casmax,2025-01-01T00:30:00Z,283.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
4,0-20000-0-11518,Casmax,2025-01-01T00:40:00Z,1.0,NaN,0.0,0-20000-0-11518,2025,1,Čas maxima,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045395,0-203-0-11101007001,T,2025-12-31T23:10:00Z,-3.7,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy","Praha, Brdy",862.0,13.818380,49.658890
5045396,0-203-0-11101007001,T,2025-12-31T23:20:00Z,-3.6,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy","Praha, Brdy",862.0,13.818380,49.658890
5045397,0-203-0-11101007001,T,2025-12-31T23:30:00Z,-3.6,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy","Praha, Brdy",862.0,13.818380,49.658890
5045398,0-203-0-11101007001,T,2025-12-31T23:40:00Z,-3.6,NaN,0.0,0-203-0-11101007001,2025,12,Teplota,"Praha, Brdy","Praha, Brdy",862.0,13.818380,49.658890


In [83]:
print((df_weather_ext["FULL_NAME"] != df_weather_ext["WSI_NAME"]).any())
print((df_weather_ext["WSI"] != df_weather_ext["STATION"]).any())

False
False


In [116]:
### air quality chmi data

In [117]:
air_stations_meta = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_stations_metadata.csv"))

In [118]:
air_stations_meta

,id_registration,station_code,street,city,lon,lat,alt,component_code,component_name,unit
0,40555,TOFFA,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,SO2,oxid siřičitý,ug/m^3
1,40557,TOFFA,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NO2,oxid dusičitý,ug/m^3
2,40560,TOFFA,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NOx,oxidy dusíku,ug/m^3
3,40559,TOFFA,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,O3,přízemní ozon,ug/m^3
4,40561,TOFFA,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,PM10,částice PM10,ug/m^3
...,...,...,...,...,...,...,...,...,...,...
479,1648406,TRYCA,Mírová,Rychvald,18.377254,49.871670,241 m,INDX,Index kvality ovzduší,NaN
480,1410468,ACHOA,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NO2,oxid dusičitý,ug/m^3
481,1410474,ACHOA,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NOx,oxidy dusíku,ug/m^3
482,1410479,ACHOA,Vejvanovského,Praha 4,14.517450,50.030170,300 m,PM10,částice PM10,ug/m^3


In [ ]:
air_stations_meta = air_stations_meta[air_stations_meta['locality_name'].str.contains('Praha', case=False, na=False)]


In [120]:
air_stations_meta

,id_registration,station_code,street,city,lon,lat,alt,component_code,component_name,unit
66,41157,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,NO2,oxid dusičitý,ug/m^3
67,41156,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,NOx,oxidy dusíku,ug/m^3
68,41158,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,CO,oxid uhelnatý,ug/m^3
69,867419,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,PM2_5,"jemné částice PM2,5",ug/m^3
70,867416,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,PM10,částice PM10,ug/m^3
71,1648574,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,INDX,Index kvality ovzduší,NaN
315,40238,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,O3,přízemní ozon,ug/m^3
316,40237,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,PM10,částice PM10,ug/m^3
317,1648379,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,INDX,Index kvality ovzduší,NaN
433,1317399,ABREA,Šlikova,Praha 6,14.380116,50.084385,300 m,NO2,oxid dusičitý,ug/m^3


In [121]:
### this is only for one import file, i.e. one day

df_air_qual = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_stations_data.csv"))

In [122]:
df_air_qual

,idRegistration,startTime,idValueType,value
0,10221,2026-05-01T12:00:00Z,6,-5003.0
1,40237,2026-05-01T12:00:00Z,8,3.5
2,40238,2026-05-01T12:00:00Z,8,110.5
3,40244,2026-05-01T12:00:00Z,8,105.7
4,40257,2026-05-01T12:00:00Z,8,1.3
...,...,...,...,...
467,2181672,2026-05-01T12:00:00Z,8,8.2
468,2181676,2026-05-01T12:00:00Z,8,4.0
469,2187304,2026-05-01T12:00:00Z,8,9.7
470,2187315,2026-05-01T12:00:00Z,8,3.4


In [123]:
### match with metadata based on idregistration

df_air_qual = df_air_qual.rename(columns={
    "idRegistration": "id_registration"
}
)

df_air_qual["id_registration"] = df_air_qual["id_registration"].astype(str)
air_stations_meta["id_registration"] = air_stations_meta["id_registration"].astype(str)

In [124]:
# join

df_air_qual = df_air_qual.merge(air_stations_meta, on="id_registration", how="right")

In [125]:
df_air_qual

,id_registration,startTime,idValueType,value,station_code,street,city,lon,lat,alt,component_code,component_name,unit
0,41157,2026-05-01T12:00:00Z,8,33.3,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,NO2,oxid dusičitý,ug/m^3
1,41156,2026-05-01T12:00:00Z,8,56.6,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,NOx,oxidy dusíku,ug/m^3
2,41158,2026-05-01T12:00:00Z,8,549.0,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,CO,oxid uhelnatý,ug/m^3
3,867419,2026-05-01T12:00:00Z,8,6.4,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,PM2_5,"jemné částice PM2,5",ug/m^3
4,867416,2026-05-01T12:00:00Z,8,14.5,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,PM10,částice PM10,ug/m^3
5,1648574,2026-05-01T12:00:00Z,148,4.0,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,INDX,Index kvality ovzduší,NaN
6,40238,2026-05-01T12:00:00Z,8,110.5,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,O3,přízemní ozon,ug/m^3
7,40237,2026-05-01T12:00:00Z,8,3.5,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,PM10,částice PM10,ug/m^3
8,1648379,2026-05-01T12:00:00Z,148,3.0,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,INDX,Index kvality ovzduší,NaN
9,1317399,2026-05-01T12:00:00Z,8,6.5,ABREA,Šlikova,Praha 6,14.380116,50.084385,300 m,NO2,oxid dusičitý,ug/m^3


In [ ]:
### golemio air quality 

#### air quality metadata processing

# station_cols = {
#     'geometry.coordinates': 'coordinates', 
#     'properties.id': 'id', 
#     'properties.name': 'name', 
#     'properties.district': 'district', 
#     'properties.measurement.components.type': 'components'
# }

# air_quality_stations = df[station_cols.keys()]

# air_quality_stations = air_quality_stations.rename(columns=station_cols)

# air_quality_stations = (
#     air_quality_stations.groupby('id', as_index=False)
#     .agg({
#         'coordinates': 'first',
#         'name': 'first',
#         'district': 'first',
#         'components': list
#     })
# )

# air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
# air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
# air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")


In [ ]:

#air quality stations dictionary

# air_stat_dict = dict(
#     zip(
#         air_quality_stations['id'].astype(str),
#         air_quality_stations['name'].astype(str)  
#     )
# )

In [ ]:
### loading the dictionaries


def load_wsi_dict(path="data/raw/wsi_dict.csv"):
    df = pd.read_csv(path, dtype=str)
    return dict(df.values)

def load_chmi_vars(path="data/raw/chmi_vars.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)